# JobShield v3 · PhoBERT Kiểu A (1 encoder, 2 đầu: Cổng + 8 mã)
Chọn **Runtime → Change runtime type → T4 GPU**, bấm **Run all**, tải lên `jobshield-v3-train-val.zip` khi được hỏi và giữ phiên mở đến khi tải xong file `.zip` kết quả (thả vào `JobShield/models/v3/`).


In [ ]:
import torch
assert torch.cuda.is_available(), "Chọn Runtime → Change runtime type → T4 GPU rồi chạy lại."
%pip install -q "transformers==4.57.6" "accelerate==1.12.0" "py-vncorenlp==0.1.4"

MODEL_ID = "vinai/phobert-base"
MAX_LENGTH = 256
SEED = 42
EPOCHS = 6
LR = 2e-5
WEIGHT_DECAY = 0.01
MAX_POS_WEIGHT = 4.0
TRAIN_BATCH_SIZE = 16
GRAD_ACCUM = 1
EVAL_BATCH_SIZE = 32
ENABLE_ANTI_SHORTCUT = True  # Đổi thành False để chạy lượt đối chứng tắt chống học vẹt


In [ ]:
import hashlib, io, json, tempfile, zipfile
from pathlib import Path
from google.colab import files

root = Path(tempfile.mkdtemp(prefix="jobshield-v3-", dir="/content"))
uploaded = files.upload()
assert len(uploaded) == 1, "Chọn đúng 1 tệp jobshield-v3-train-val.zip."
raw = next(iter(uploaded.values()))
input_sha = hashlib.sha256(raw).hexdigest()
assert input_sha == "99d9ea1e6f494c902c9cfbd39a20f344b6af443aaaefec372e2286c1f9c70928", f"Sai gói dữ liệu! SHA-256 nhận được: {input_sha}"

with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    train_rows = [json.loads(line) for line in archive.read("train.jsonl").splitlines()]
    val_rows = [json.loads(line) for line in archive.read("validation.jsonl").splitlines()]

assert (len(train_rows), len(val_rows)) == (1035, 69)
assert all(r["overall_label"] in {"suspicious", "not_detected"} for r in train_rows + val_rows)
codes = sorted({f["code"] for r in train_rows for f in r["output"]["findings"]})
assert codes == ["G01", "G02", "G03", "G04", "G05", "G06", "G07", "G08"]
print(f"Đã nạp {len(train_rows)} mẫu train và {len(val_rows)} mẫu validation. Nhãn: {codes}")


In [ ]:
import random
import re
from typing import Any, Sequence

CODES = ("G01", "G02", "G03", "G04", "G05", "G06", "G07", "G08")
PLACEHOLDER_RE = re.compile(r"\[\s*[A-Z0-9_]+\s*\]")

_RAW_PLACEHOLDER_POOLS = {
    "[PHONE]": ("0912345678", "0987654321", "0389123456", "số điện thoại cá nhân"),
    "[URL]": ("https://tuyen-dung.example.vn", "đường dẫn đăng ký", "link trang web"),
    "[LINK]": ("đường dẫn riêng", "link biểu mẫu", "https://portal.example.com"),
    "[ZALO_URL]": ("link Zalo cá nhân", "nhóm Zalo tuyển dụng"),
    "[CONTACT]": ("Zalo cá nhân", "Telegram riêng", "liên hệ trực tiếp"),
    "[MONEY]": ("500.000đ", "300k", "2 triệu đồng", "1.500.000đ"),
    "[NUMBER]": ("500", "300", "24", "10"),
    "[BANK_ACCOUNT]": ("tài khoản ngân hàng cá nhân", "1903456789012", "số tài khoản nhận tiền"),
    "[ACCOUNT]": ("tài khoản ngân hàng", "tài khoản ví điện tử", "tài khoản cá nhân"),
    "[EMAIL]": ("hr@example.vn", "tuyendung@example.com"),
    "[ADDRESS]": ("Quận 1, TP.HCM", "Cầu Giấy, Hà Nội", "địa chỉ văn phòng"),
    "[NAME]": ("Minh", "Lan", "Hùng"),
    "[CODE]": ("849201", "123456"),
    "[REDACTED]": ("thông tin đã ẩn", "nội dung liên hệ"),
}


def _to_segmented_variant(text: str) -> str:
    if text.startswith("http") or text.isdigit() or "@" in text:
        return text
    return text.replace(" ", "_")


def normalize_sentences(row_or_sentences, doc_type=None):
    if isinstance(row_or_sentences, dict):
        inp = row_or_sentences.get("input", row_or_sentences)
        raw_sentences = inp["sentences"]
        resolved_doc_type = doc_type or inp.get("type", "unknown")
    else:
        raw_sentences = row_or_sentences
        resolved_doc_type = doc_type
    return [
        {
            "id": str(s["id"]),
            "type": s.get("type") or resolved_doc_type or "unknown",
            "speaker": s.get("speaker") or "unknown",
            "text": str(s["text"]),
        }
        for s in raw_sentences
    ]


def augment_sentences(sentences, rng, enable=True, prefix_dropout_p=0.5, placeholder_sub_p=0.7, word_segmented=False):
    if not enable:
        return [dict(s) for s in sentences]
    drop_channel = rng.random() < prefix_dropout_p

    def _replace_placeholder(match):
        key = re.sub(r"\s+", "", match.group(0))
        if rng.random() >= placeholder_sub_p:
            return key
        pool = _RAW_PLACEHOLDER_POOLS.get(key, ("thông tin",))
        choice = rng.choice(pool)
        return _to_segmented_variant(choice) if word_segmented else choice

    return [
        {
            "id": s["id"],
            "type": "unknown" if drop_channel else s["type"],
            "speaker": "unknown" if rng.random() < prefix_dropout_p else s["speaker"],
            "text": PLACEHOLDER_RE.sub(_replace_placeholder, s["text"]),
        }
        for s in sentences
    ]


def encode_sentences(sentences, tokenizer, max_length=256, segmenter=None, presegmented_texts=None):
    if presegmented_texts is not None:
        texts = list(presegmented_texts)
    elif segmenter is not None:
        texts = [" ".join(segmenter.word_segment(s["text"])) for s in sentences]
    else:
        texts = [s["text"] for s in sentences]

    parts = [
        tokenizer.encode(f"{s['type']} {s['speaker']}: {text}", add_special_tokens=False, verbose=False)
        for s, text in zip(sentences, texts, strict=True)
    ]
    sep = [tokenizer.sep_token_id]
    special_overhead = len(tokenizer.build_inputs_with_special_tokens([]))
    inner_budget = max_length - special_overhead - 2
    stream, starts = [], []
    for part in parts:
        starts.append(len(stream))
        stream.extend(part + sep)

    features = []
    for start, part in zip(starts, parts, strict=True):
        focus = part[:inner_budget]
        available = inner_budget - len(focus)
        after = start + len(part) + 1
        left = min(start, available // 2)
        right = min(len(stream) - after, available - left)
        left = min(start, available - right)
        tokens = stream[start - left : start] + sep + focus + sep + stream[after : after + right]
        input_ids = tokenizer.build_inputs_with_special_tokens(tokens)
        assert len(input_ids) <= max_length
        features.append({"input_ids": input_ids})
    return features


def encode_document_head_tail(sentences, tokenizer, max_length=256, segmenter=None, presegmented_texts=None):
    if presegmented_texts is not None:
        texts = list(presegmented_texts)
    elif segmenter is not None:
        texts = [" ".join(segmenter.word_segment(s["text"])) for s in sentences]
    else:
        texts = [s["text"] for s in sentences]

    sep_id = tokenizer.sep_token_id
    special_overhead = len(tokenizer.build_inputs_with_special_tokens([]))
    inner_budget = max_length - special_overhead
    stream = []
    for idx, (s, text) in enumerate(zip(sentences, texts, strict=True)):
        if idx > 0:
            stream.append(sep_id)
        stream.extend(tokenizer.encode(f"{s['type']} {s['speaker']}: {text}", add_special_tokens=False, verbose=False))

    if len(stream) > inner_budget:
        head_len = inner_budget // 2
        tail_len = inner_budget - head_len - 1
        stream = stream[:head_len] + [sep_id] + (stream[-tail_len:] if tail_len > 0 else [])
    input_ids = tokenizer.build_inputs_with_special_tokens(stream)
    assert len(input_ids) <= max_length
    return {"input_ids": input_ids}


def decode_two_gate_predictions(doc_gate_score, sentence_ids, sentence_code_scores, codes, tau_gate, code_thresholds):
    if doc_gate_score < tau_gate:
        return {"label": "not_detected", "findings": []}
    evidence_by_code = {}
    best_pair = None
    for sid, scores in zip(sentence_ids, sentence_code_scores, strict=True):
        for code, score in zip(codes, scores, strict=True):
            val = float(score)
            if best_pair is None or val > best_pair[0]:
                best_pair = (val, code, sid)
            if val >= code_thresholds[code]:
                evidence_by_code.setdefault(code, []).append(sid)
    if not evidence_by_code and best_pair is not None:
        _, best_code, best_sid = best_pair
        evidence_by_code[best_code] = [best_sid]
    findings = [{"code": c, "evidence_sentence_ids": sids} for c, sids in sorted(evidence_by_code.items())]
    return {"label": "suspicious" if findings else "not_detected", "findings": findings}


def _f1(tp, fp, fn):
    d = 2 * tp + fp + fn
    return (2.0 * tp / d) if d > 0 else 0.0


def optimize_two_gate_thresholds(val_rows, doc_gate_scores, sent_code_scores_by_doc, codes=CODES, max_false_alarms=2):
    truth_pos = {r["sample_id"] for r in val_rows if r["overall_label"] == "suspicious"}
    gold_pairs = {
        (r["sample_id"], f["code"], sid)
        for r in val_rows
        for f in r["output"]["findings"]
        for sid in f["evidence_sentence_ids"]
    }
    best_gate, best_fallback = None, None
    for step in range(5, 96, 2):
        tau = round(step / 100.0, 2)
        pred_pos = {r["sample_id"] for r, g in zip(val_rows, doc_gate_scores, strict=True) if g >= tau}
        tp, fp, fn = len(pred_pos & truth_pos), len(pred_pos - truth_pos), len(truth_pos - pred_pos)
        f1 = _f1(tp, fp, fn)
        recall = tp / max(1, len(truth_pos))
        precision = tp / max(1, tp + fp)
        cand = {"tau_gate": tau, "tp": tp, "fp": fp, "fn": fn, "precision": round(precision, 4), "recall": round(recall, 4), "f1": round(f1, 4)}
        if fp <= max_false_alarms:
            key = (f1, recall, -fp, -abs(tau - 0.5))
            if best_gate is None or key > best_gate["_key"]:
                best_gate = {**cand, "_key": key}
        fb_key = (-fp, f1, recall, -abs(tau - 0.5))
        if best_fallback is None or fb_key > best_fallback["_key"]:
            best_fallback = {**cand, "_key": fb_key}

    chosen_gate = best_gate if best_gate is not None else best_fallback
    tau_gate = float(chosen_gate["tau_gate"])
    code_thresholds = {}
    for code_idx, code in enumerate(codes):
        gold_c = {x for x in gold_pairs if x[1] == code}
        best_c_key, best_tau_c = None, 0.5
        for step in range(15, 91, 5):
            tau_c = round(step / 100.0, 2)
            pred_c = set()
            for row, g_score, s_scores in zip(val_rows, doc_gate_scores, sent_code_scores_by_doc, strict=True):
                if g_score < tau_gate:
                    continue
                for sent, scores in zip(row["input"]["sentences"], s_scores, strict=True):
                    if float(scores[code_idx]) >= tau_c:
                        pred_c.add((row["sample_id"], code, sent["id"]))
            tp, fp, fn = len(pred_c & gold_c), len(pred_c - gold_c), len(gold_c - pred_c)
            key = (_f1(tp, fp, fn), -abs(tau_c - 0.5), -fp)
            if best_c_key is None or key > best_c_key:
                best_c_key, best_tau_c = key, tau_c
        code_thresholds[code] = best_tau_c

    pred_pairs, pred_docs, exact_correct = set(), set(), 0
    for row, g_score, s_scores in zip(val_rows, doc_gate_scores, sent_code_scores_by_doc, strict=True):
        sids = [s["id"] for s in row["input"]["sentences"]]
        out = decode_two_gate_predictions(g_score, sids, s_scores, codes, tau_gate, code_thresholds)
        sid_codes = {(f["code"], sid) for f in out["findings"] for sid in f["evidence_sentence_ids"]}
        gold_doc_pairs = {(f["code"], sid) for f in row["output"]["findings"] for sid in f["evidence_sentence_ids"]}
        if out["label"] == "suspicious":
            pred_docs.add(row["sample_id"])
            for c, sid in sid_codes:
                pred_pairs.add((row["sample_id"], c, sid))
        if out["label"] == row["overall_label"] and sid_codes == gold_doc_pairs:
            exact_correct += 1

    chosen_gate.pop("_key", None)
    return {
        "tau_gate": tau_gate,
        "code_thresholds": code_thresholds,
        "validation_summary": {
            **chosen_gate,
            "n": len(val_rows),
            "label_correct": len(val_rows) - len(pred_docs ^ truth_pos),
            "exact_correct": exact_correct,
            "evidence_f1": round(_f1(len(pred_pairs & gold_pairs), len(pred_pairs - gold_pairs), len(gold_pairs - pred_pairs)), 4),
            "max_false_alarms_policy": max_false_alarms,
        },
    }


In [ ]:
import copy, math, random, shutil
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, DataCollatorWithPadding, get_linear_schedule_with_warmup, set_seed

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
import py_vncorenlp
segmenter_dir = root / "vncorenlp"
segmenter_dir.mkdir(exist_ok=True)
py_vncorenlp.download_model(save_dir=str(segmenter_dir))
segmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=str(segmenter_dir))

def _prepare_docs(rows):
    prepared = []
    for row in rows:
        norm_sents = normalize_sentences(row)
        preseg = [" ".join(segmenter.word_segment(s["text"])) for s in norm_sents]
        gold = {(f["code"], sid) for f in row["output"]["findings"] for sid in f["evidence_sentence_ids"]}
        sent_labels = []
        gate_labels = []
        for s in norm_sents:
            vec = [1.0 if (c, s["id"]) in gold else 0.0 for c in CODES]
            sent_labels.append(vec)
            gate_labels.append(1.0 if any(vec) else 0.0)
        # Lưu bản text đã tách từ để augment nhanh không cần gọi lại VnCoreNLP
        base_sents = [{**s, "text": txt} for s, txt in zip(norm_sents, preseg, strict=True)]
        prepared.append({"row": row, "sents": base_sents, "gate_labels": gate_labels, "code_labels": sent_labels})
    return prepared

train_docs = _prepare_docs(train_rows)
val_docs = _prepare_docs(val_rows)

def build_sentence_samples(docs, rng, enable_aug):
    items = []
    for doc in docs:
        sents = augment_sentences(doc["sents"], rng, enable=enable_aug, word_segmented=True)
        feats = encode_sentences(sents, tokenizer, max_length=MAX_LENGTH, presegmented_texts=[s["text"] for s in sents])
        for feat, g_lbl, c_lbl in zip(feats, doc["gate_labels"], doc["code_labels"], strict=True):
            items.append({
                "input_ids": feat["input_ids"],
                "attention_mask": [1] * len(feat["input_ids"]),
                "gate_label": g_lbl,
                "code_label": c_lbl,
            })
    return items

class SentenceDataset(Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        return self.items[idx]

collator = DataCollatorWithPadding(tokenizer)
def collate_fn(batch):
    gate_y = torch.tensor([b["gate_label"] for b in batch], dtype=torch.float32)
    code_y = torch.tensor([b["code_label"] for b in batch], dtype=torch.float32)
    padded = collator([{"input_ids": b["input_ids"], "attention_mask": b["attention_mask"]} for b in batch])
    padded["gate_label"] = gate_y
    padded["code_label"] = code_y
    return padded

class TwoHeadClassifier(torch.nn.Module):
    def __init__(self, model_id, num_codes=8):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_id)
        h = self.encoder.config.hidden_size
        self.dropout = torch.nn.Dropout(0.1)
        self.gate_head = torch.nn.Linear(h, 1)
        self.code_head = torch.nn.Linear(h, num_codes)
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(out.last_hidden_state[:, 0, :])
        return self.gate_head(pooled).squeeze(-1), self.code_head(pooled)

# Tính pos_weight làm dịu có trần MAX_POS_WEIGHT
raw_train_items = build_sentence_samples(train_docs, random.Random(SEED), enable_aug=False)
g_pos = sum(x["gate_label"] for x in raw_train_items)
g_pw = torch.tensor([min(MAX_POS_WEIGHT, math.sqrt((len(raw_train_items) - g_pos) / max(1.0, g_pos)))], device=device)
c_counts = np.sum([x["code_label"] for x in raw_train_items], axis=0)
c_pw = torch.tensor([min(MAX_POS_WEIGHT, math.sqrt((len(raw_train_items) - cnt) / max(1.0, cnt))) for cnt in c_counts], dtype=torch.float32, device=device)

val_items = build_sentence_samples(val_docs, random.Random(SEED), enable_aug=False)
val_loader = DataLoader(SentenceDataset(val_items), batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

model = TwoHeadClassifier(MODEL_ID, num_codes=len(CODES)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
steps_per_epoch = math.ceil(len(raw_train_items) / (TRAIN_BATCH_SIZE * GRAD_ACCUM))
total_steps = steps_per_epoch * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=max(1, int(0.1 * total_steps)), num_training_steps=total_steps)
scaler = torch.amp.GradScaler("cuda")

@torch.inference_mode()
def evaluate_val(net):
    net.eval()
    g_probs, c_probs = [], []
    for batch in val_loader:
        g_log, c_log = net(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        g_probs.extend(torch.sigmoid(g_log.float()).cpu().tolist())
        c_probs.extend(torch.sigmoid(c_log.float()).cpu().tolist())
    doc_gate_scores, sent_code_scores_by_doc = [], []
    idx = 0
    for doc in val_docs:
        n_s = len(doc["sents"])
        doc_gate_scores.append(max(g_probs[idx : idx + n_s]))
        sent_code_scores_by_doc.append(c_probs[idx : idx + n_s])
        idx += n_s
    return doc_gate_scores, sent_code_scores_by_doc

best_score, best_state = None, None
for epoch in range(1, EPOCHS + 1):
    model.train()
    ep_items = build_sentence_samples(train_docs, random.Random(SEED + epoch), enable_aug=ENABLE_ANTI_SHORTCUT)
    ep_loader = DataLoader(SentenceDataset(ep_items), batch_size=TRAIN_BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    for step, batch in enumerate(ep_loader, 1):
        with torch.amp.autocast("cuda"):
            g_log, c_log = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            loss_g = torch.nn.functional.binary_cross_entropy_with_logits(g_log.float(), batch["gate_label"].to(device), pos_weight=g_pw)
            loss_c = torch.nn.functional.binary_cross_entropy_with_logits(c_log.float(), batch["code_label"].to(device), pos_weight=c_pw)
            loss = (loss_g + loss_c) / GRAD_ACCUM
        scaler.scale(loss).backward()
        running_loss += float(loss.item()) * GRAD_ACCUM
        if step % GRAD_ACCUM == 0 or step == len(ep_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

    d_gate, s_codes = evaluate_val(model)
    tuned = optimize_two_gate_thresholds(val_rows, d_gate, s_codes, CODES, max_false_alarms=2)
    vs = tuned["validation_summary"]
    score_key = (vs["f1"], vs["evidence_f1"], -vs["fp"])
    print(f"Epoch {epoch}/{EPOCHS} | loss={running_loss/len(ep_loader):.4f} | val_f1={vs['f1']:.4f} | ev_f1={vs['evidence_f1']:.4f} | exact={vs['exact_correct']}/{vs['n']} | fp={vs['fp']}")
    if best_score is None or score_key > best_score:
        best_score = score_key
        best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})

model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
print("Đã chọn checkpoint tốt nhất với key:", best_score)


In [ ]:
val_gate_scores, val_sent_scores = evaluate_val(model)
tuned = optimize_two_gate_thresholds(val_rows, val_gate_scores, val_sent_scores, CODES, max_false_alarms=2)

run_name = "phobert-v3-typeA" if ENABLE_ANTI_SHORTCUT else "phobert-v3-typeA-no-antishortcut"
out = root / "result"
ckpt_dir = out / "checkpoint"
(ckpt_dir / "base_model").mkdir(parents=True, exist_ok=True)
model.encoder.save_pretrained(ckpt_dir / "base_model")
tokenizer.save_pretrained(ckpt_dir)
torch.save({"gate_head": model.gate_head.state_dict(), "code_head": model.code_head.state_dict()}, ckpt_dir / "two_head_weights.pt")

meta = {
    "model_name": run_name,
    "model_type": "transformer_typeA",
    "architecture": "typeA",
    "backbone": MODEL_ID,
    "max_length": 256,
    "word_segmentation": True,
    "anti_shortcut": bool(ENABLE_ANTI_SHORTCUT),
    "dataset_version": "v3",
    "seed": SEED,
    "codes": list(CODES),
    "train_samples": len(train_rows),
    "validation_samples": len(val_rows),
    "dataset_sha256": {"train_val_zip": input_sha},
    "reload_verified": True,
}
(out / "model_meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
(out / "thresholds.json").write_text(json.dumps({"tau_gate": tuned["tau_gate"], "code_thresholds": tuned["code_thresholds"]}, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
(out / "validation.json").write_text(json.dumps({**meta, **tuned}, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
with (out / "validation_scores.jsonl").open("w", encoding="utf-8") as fh:
    for row, g_sc, s_sc in zip(val_rows, val_gate_scores, val_sent_scores, strict=True):
        print(json.dumps({"sample_id": row["sample_id"], "doc_gate_score": g_sc, "sentence_scores": s_sc}), file=fh)

# Xác minh reload từ đĩa ở float32 trên CPU
reloaded = TwoHeadClassifier(ckpt_dir / "base_model", num_codes=len(CODES)).eval()
heads = torch.load(ckpt_dir / "two_head_weights.pt", map_location="cpu", weights_only=True)
reloaded.gate_head.load_state_dict(heads["gate_head"])
reloaded.code_head.load_state_dict(heads["code_head"])
re_tok = AutoTokenizer.from_pretrained(ckpt_dir)
re_gate, re_sent = [], []
with torch.inference_mode():
    for doc in val_docs:
        feats = encode_sentences(doc["sents"], re_tok, max_length=MAX_LENGTH, presegmented_texts=[s["text"] for s in doc["sents"]])
        batch = re_tok.pad(feats, padding=True, return_tensors="pt")
        g_l, c_l = reloaded(batch["input_ids"], batch["attention_mask"])
        re_gate.append(max(torch.sigmoid(g_l).tolist()))
        re_sent.append(torch.sigmoid(c_l).tolist())
np.testing.assert_allclose(re_gate, val_gate_scores, rtol=1e-4, atol=1e-4)
for r_s, v_s in zip(re_sent, val_sent_scores, strict=True):
    np.testing.assert_allclose(r_s, v_s, rtol=1e-4, atol=1e-4)
print("Reload verified! Validation summary:", json.dumps(tuned["validation_summary"], ensure_ascii=False))
archive_path = shutil.make_archive(str(root / f"jobshield-{run_name}-result"), "zip", out)
print("Archive:", archive_path)
files.download(archive_path)
